In [1]:
import os 
from dotenv import load_dotenv

In [2]:
from langchain_community.document_loaders import TextLoader

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [4]:
load_dotenv()

True

In [5]:
groq_key =os.getenv("GROQ_API_KEY")
jina_key =os.getenv("JINA_API_KEY")


In [6]:
file_path = os.path.join("data","attendance_wfh_policy.txt")

In [7]:
loader = TextLoader(file_path,encoding="utf-8")

In [8]:
document = loader.load()
print("data loaded")
print("="*25)

data loaded


In [9]:
print(document)

[Document(metadata={'source': 'data\\attendance_wfh_policy.txt'}, page_content='HR POLICY DOCUMENT Document: Attendance and Work From Home Policy\nVersion: 1.0 Effective date: 1 January 2026 Company: Example\nTechnologies Pvt. Ltd.\xa0(Fictional)\n\nIMPORTANT This is fictional sample content created for software testing.\nIt is not legal advice and does not represent a real company’s policy.\n\n1.  WORKING HOURS The standard work schedule is Monday to Friday, 9:30\n    AM to 6:30 PM, with a one-hour lunch break. Individual teams may\n    have approved alternate schedules.\n\n2.  ATTENDANCE Employees must record their start and end times using the\n    company attendance system. If an employee forgets to record\n    attendance, they should submit a correction request within two\n    working days and obtain manager approval.\n\n3.  LATE ARRIVAL Employees should inform their manager if they expect to\n    arrive late. Repeated late arrivals may be reviewed by the manager\n    and HR under

In [10]:
len(document)

1

In [11]:
print(document[0].page_content)


HR POLICY DOCUMENT Document: Attendance and Work From Home Policy
Version: 1.0 Effective date: 1 January 2026 Company: Example
Technologies Pvt. Ltd. (Fictional)

IMPORTANT This is fictional sample content created for software testing.
It is not legal advice and does not represent a real company’s policy.

1.  WORKING HOURS The standard work schedule is Monday to Friday, 9:30
    AM to 6:30 PM, with a one-hour lunch break. Individual teams may
    have approved alternate schedules.

2.  ATTENDANCE Employees must record their start and end times using the
    company attendance system. If an employee forgets to record
    attendance, they should submit a correction request within two
    working days and obtain manager approval.

3.  LATE ARRIVAL Employees should inform their manager if they expect to
    arrive late. Repeated late arrivals may be reviewed by the manager
    and HR under the company’s attendance process.

4.  WORK FROM HOME ELIGIBILITY Employees may request up to 8
    

In [12]:
print(document[0].metadata)


{'source': 'data\\attendance_wfh_policy.txt'}


In [13]:
print(f"total no. of character {len(document[0].page_content)}")

total no. of character 1793


``` this is a way of chunking 

In [14]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size =400,
    chunk_overlap= 40
)
# token based chinking 
# TokenTextSplitter(
#     chunk_size=200,
#     chunk_overlap=20
# )  Useful because LLMs process tokens, not characters.

chunks = splitter.split_documents(document)

print(chunks)

[Document(metadata={'source': 'data\\attendance_wfh_policy.txt'}, page_content='HR POLICY DOCUMENT Document: Attendance and Work From Home Policy\nVersion: 1.0 Effective date: 1 January 2026 Company: Example\nTechnologies Pvt. Ltd.\xa0(Fictional)\n\nIMPORTANT This is fictional sample content created for software testing.\nIt is not legal advice and does not represent a real company’s policy.'), Document(metadata={'source': 'data\\attendance_wfh_policy.txt'}, page_content='1.  WORKING HOURS The standard work schedule is Monday to Friday, 9:30\n    AM to 6:30 PM, with a one-hour lunch break. Individual teams may\n    have approved alternate schedules.'), Document(metadata={'source': 'data\\attendance_wfh_policy.txt'}, page_content='2.  ATTENDANCE Employees must record their start and end times using the\n    company attendance system. If an employee forgets to record\n    attendance, they should submit a correction request within two\n    working days and obtain manager approval.'), Docu

In [15]:
print(len(chunks))

7


In [16]:
print(chunks[0].page_content)

HR POLICY DOCUMENT Document: Attendance and Work From Home Policy
Version: 1.0 Effective date: 1 January 2026 Company: Example
Technologies Pvt. Ltd. (Fictional)

IMPORTANT This is fictional sample content created for software testing.
It is not legal advice and does not represent a real company’s policy.


In [17]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)


In [19]:
from langchain_community.vectorstores import FAISS

In [20]:
vector_store=FAISS.from_documents(chunks, embeddings)

In [21]:
print("chunks are stored ", vector_store.index.ntotal)

chunks are stored  7


In [22]:
tq = "how many leaves I will get ?"

top_match = vector_store.similarity_search(tq,k=2)

for i, match in enumerate(top_match,start=1):
    print(f"----match{i}-----")
    print(match.page_content)


----match1-----
4.  WORK FROM HOME ELIGIBILITY Employees may request up to 8
    work-from-home days per calendar month, subject to role suitability,
    team requirements, and manager approval. This is a fictional company
    rule for testing.
----match2-----
1.  WORKING HOURS The standard work schedule is Monday to Friday, 9:30
    AM to 6:30 PM, with a one-hour lunch break. Individual teams may
    have approved alternate schedules.


In [23]:
top_match = vector_store.similarity_search_with_score(tq, k=2)

for doc, score in top_match:
    print("Score:", score)
    print("Content:", doc.page_content)
    print("-------------------")

Score: 0.6811969
Content: 4.  WORK FROM HOME ELIGIBILITY Employees may request up to 8
    work-from-home days per calendar month, subject to role suitability,
    team requirements, and manager approval. This is a fictional company
    rule for testing.
-------------------
Score: 0.690215
Content: 1.  WORKING HOURS The standard work schedule is Monday to Friday, 9:30
    AM to 6:30 PM, with a one-hour lunch break. Individual teams may
    have approved alternate schedules.
-------------------


In [24]:
from langchain_groq import ChatGroq

In [25]:
llm = ChatGroq(
    model= "openai/gpt-oss-20b",
    temperature=0
)


In [26]:
test_response = llm.invoke("hey , is learning agentic ai  hard ?")

In [28]:
print(test_response)

# test_response.content 

content='**Short answer:**  \nLearning agentic AI isn’t “easy” in the sense that you can pick it up in a weekend, but it’s also not a death‑knell. If you already know some programming, linear algebra, and basic machine learning, you’ll find the learning curve manageable—just a bit steep at first. If you’re brand‑new to AI, it will feel like a marathon, but with a clear roadmap and the right resources, you can make steady progress.\n\n---\n\n## What is “agentic AI”?\n\nAgentic AI usually refers to systems that can **act autonomously** in an environment to achieve goals. Think of:\n\n- **Reinforcement Learning (RL)** agents that learn policies by trial‑and‑error.  \n- **Planning & Decision‑Making** agents that reason about future actions.  \n- **Multi‑agent systems** where several agents interact.  \n- **Goal‑driven dialogue agents** that pursue user‑specified objectives.\n\nIn short, it’s the intersection of AI, control theory, and sometimes game theory.\n\n---\n\n## Why it feels hard\n

In [31]:
from langchain.agents import create_agent

making of tool 

In [64]:
retriever = vector_store.as_retriever(search_kwargs={"k" :2})

def search_policy_tool(question:str)->str:
    """search for hr policy and leaves """
    matching_chunk = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunk )

now we will gonna create a agent 

In [65]:
hr_assistant = create_agent(
    model =llm ,
    tools=[search_policy_tool],
    system_prompt = """ act like an hr and answer politely and if answer is not in the pdf then tell it's not in our policy"""


)

In [66]:
def ask_assistant(question : str)-> str:
    print("quesion:", question)
    print("="*50)
    response = hr_assistant.invoke( [{"messages" :{"role": "user" , "content":question}}])
    answer = response["messages"][-1].content
    print("+"*60)
    print("answer:", answer )
    return answer 

    

In [67]:
response = hr_assistant.invoke( {
    "messages" :[{"role": "user" , 
                 "content":"how many hours i will have to work"}
                 ]})

In [73]:
(response)

{'messages': [HumanMessage(content='how many hours i will have to work', additional_kwargs={}, response_metadata={}, id='1f028ac6-4831-4cd7-9813-de5b92ab708e'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to answer politely, as HR. We need to check if the answer is in the PDF. We have a tool to search policy. We should call search_policy_tool with question "how many hours i will have to work". Then respond accordingly.', 'tool_calls': [{'id': 'fc_eb43a7a7-c4ce-4696-9ee1-9bd2423dd14c', 'function': {'arguments': '{"question":"how many hours i will have to work"}', 'name': 'search_policy_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 83, 'prompt_tokens': 155, 'total_tokens': 238, 'completion_time': 0.083831007, 'completion_tokens_details': {'reasoning_tokens': 52}, 'prompt_time': 0.026895784, 'prompt_tokens_details': None, 'queue_time': 0.255945319, 'total_time': 0.110726791}, 'model_name': 'openai/gpt-oss-20b', 'system_

In [75]:
print(response["messages"][-2].content)

1.  WORKING HOURS The standard work schedule is Monday to Friday, 9:30
    AM to 6:30 PM, with a one-hour lunch break. Individual teams may
    have approved alternate schedules.

4.  WORK FROM HOME ELIGIBILITY Employees may request up to 8
    work-from-home days per calendar month, subject to role suitability,
    team requirements, and manager approval. This is a fictional company
    rule for testing.
